## Buyer Search Pipeline ? Integration Test

?? Requires BAAI/bge-m3 embedding model (teammate's machine)

This notebook verifies the buyer search pipeline end-to-end using live MongoDB data and an in-memory query fixture generated from `process_query()`.

- Teammate-owned work: dataset preparation, LLM generation, embedding generation, indexing, and seller-side insertion.
- This-machine work: MongoDB retrieval, aggregation, hybrid ranking, result formatting, and pipeline smoke testing.
- No fixture JSON file is required. The notebook builds the fixture from `TEST_QUERY` at runtime.


In [ ]:
# Cell 2 — Setup & Connection
from pathlib import Path
import json
import pymongo

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    get_ipython().run_line_magic("cd", str(ROOT.parent))
    ROOT = Path.cwd()

from src.mongodb import get_items_collection, get_retrieval_units_collection, ping_mongodb

items = get_items_collection()
retrieval_units = get_retrieval_units_collection()

ping = ping_mongodb()
print("MongoDB connection:", "PASS" if ping.get("ok") else "FAIL")
if not ping.get("ok"):
    raise RuntimeError(ping.get("error", "MongoDB ping failed"))

collection_counts = {
    "total_items": items.count_documents({}),
    "total_retrieval_units": retrieval_units.count_documents({}),
    "hype_question_units_with_embedding": retrieval_units.count_documents({
        "unit_type": "hype_question",
        "embedding": {"$exists": True},
    }),
    "proposition_units_with_text_search": retrieval_units.count_documents({
        "unit_type": "proposition",
        "text_search": {"$exists": True},
    }),
    "cold_start_items": items.count_documents({"cold_start.is_cold_item": True}),
}

print("\nCollection health")
print("Metric | Count")
print("--- | ---:")
for metric, count in collection_counts.items():
    print(f"{metric} | {count}")

collection_health_pass = all(count > 0 for count in collection_counts.values())
print("\nCollection health status:", "PASS" if collection_health_pass else "FAIL")


In [ ]:
# Cell 3 ? Build Test Fixture from Query Processor
from src.query_processor import process_query

TEST_QUERY = "Fast charge samsung galaxy a14 5g"
fixture = process_query(TEST_QUERY)

fixture_preview = dict(fixture)
embedding = fixture_preview.get("query_embedding")
if isinstance(embedding, list):
    fixture_preview["query_embedding"] = embedding[:3] + ["..."]

print(f"Built fixture from TEST_QUERY: {TEST_QUERY}")
print(json.dumps(fixture_preview, indent=2, ensure_ascii=False))


In [ ]:
# Cell 4 — Run Search Pipeline
from src.search_pipeline import run_search

results = run_search(fixture, mode="unionWith", top_k=10)
pipeline_returns_results_pass = len(results) > 0

print("Search mode: unionWith")
print("Top K: 10")
print("Raw result count:", len(results))


In [ ]:
# Cell 5 — Results Table
def preview_text(value, max_len=60):
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def result_channels(result):
    channels = (result.get("debug") or {}).get("matched_channels") or []
    return channels if isinstance(channels, list) else [channels]

print("Rank | Item ID | Title (60 chars) | Score | Vector Rank | BM25 Rank | Channels | Cold Start")
print("---: | --- | --- | ---: | ---: | ---: | --- | ---")
for rank, result in enumerate(results, start=1):
    channels = result_channels(result)
    print(
        f"{rank} | "
        f"{result.get('item_id')} | "
        f"{preview_text(result.get('title'))} | "
        f"{result.get('score')} | "
        f"{result.get('rank_vector')} | "
        f"{result.get('rank_bm25')} | "
        f"{', '.join(str(channel) for channel in channels)} | "
        f"{'yes' if result.get('cold_start_note') else 'no'}"
    )

hybrid_channels_pass = any(
    "vector" in result_channels(result) and "bm25" in result_channels(result)
    for result in results
)


In [ ]:
# Cell 6 — Field Contract Check
def get_path(doc, path):
    current = doc
    for part in path.split("."):
        if not isinstance(current, dict) or part not in current:
            return False, None
        current = current[part]
    return current is not None, current

def value_preview(value, max_len=80):
    if isinstance(value, list):
        if len(value) > 6:
            return f"list(len={len(value)}, first={value[:3]})"
        return str(value)
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def print_contract_table(title, doc, fields):
    print(f"\n{title}")
    print("Field | Status | Value Preview")
    print("--- | --- | ---")
    passed = True
    for field in fields:
        exists, value = get_path(doc, field)
        if field == "embedding" and exists:
            exists = isinstance(value, list) and len(value) == 1024
        passed = passed and exists
        print(f"{field} | {'PASS' if exists else 'FAIL'} | {value_preview(value)}")
    return passed

hype_unit = retrieval_units.find_one({"unit_type": "hype_question", "embedding": {"$exists": True}})
proposition_unit = retrieval_units.find_one({"unit_type": "proposition", "text_search": {"$exists": True}})
lookup_item_id = (hype_unit or proposition_unit or {}).get("item_id")
lookup_item = items.find_one({"_id": lookup_item_id}) if lookup_item_id else None

hype_fields = [
    "item_id", "unit_type", "language", "embedding", "embedding_text", "raw_text", "aspect",
    "confidence", "category_id", "price_bucket", "in_stock", "is_cold_item",
]
proposition_fields = [
    "item_id", "unit_type", "language", "text_search", "item_title_en", "item_brand",
    "confidence", "category_id", "price_bucket", "in_stock", "is_cold_item",
]
item_fields = [
    "_id", "title_en", "brand", "price_vnd", "price_bucket", "in_stock", "image_url",
    "cold_start.is_cold_item",
]

print("Selected hype_question _id:", None if hype_unit is None else hype_unit.get("_id"))
hype_contract_pass = print_contract_table("Hype unit contract", hype_unit, hype_fields)

print("\nSelected proposition _id:", None if proposition_unit is None else proposition_unit.get("_id"))
proposition_contract_pass = print_contract_table("Proposition unit contract", proposition_unit, proposition_fields)

print("\nLookup item _id:", lookup_item_id)
item_contract_pass = print_contract_table("Item contract", lookup_item, item_fields)

field_contract_pass = hype_contract_pass and proposition_contract_pass and item_contract_pass
print("\nField contract status:", "PASS" if field_contract_pass else "FAIL")


In [ ]:
# Cell 7 ? Summary (Markdown-style)
fixture_built_from_query = bool(fixture.get("query_embedding")) and len(fixture.get("query_embedding", [])) == 1024
ready_for_demo_ui = (
    collection_health_pass
    and fixture_built_from_query
    and pipeline_returns_results_pass
    and hybrid_channels_pass
    and field_contract_pass
)

print("## Integration Test Summary")
print()
print(f"- Test query: {TEST_QUERY}")
print(f"- Fixture built with query_processor: {'PASS' if fixture_built_from_query else 'FAIL'}")
print(f"- Collection health: {'PASS' if collection_health_pass else 'FAIL'}")
print(f"- Pipeline returns results: {'PASS' if pipeline_returns_results_pass else 'FAIL'}")
print(f"- Hybrid channels working: {'PASS' if hybrid_channels_pass else 'FAIL'}")
print(f"- Field contract: {'PASS' if field_contract_pass else 'FAIL'}")
print(f"- Ready for Demo UI: {'YES' if ready_for_demo_ui else 'NO'}")
